# Per-cycle feature extraction across all four tasks

Reads HDF5 recordings, segments into cycles, computes 110-dim feature vector per cycle. Output: features.csv with all T1-T4 cycles.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
# same MIN_CYCLE_SAMPLES = 200 filter, same TRIM_FIRST_LAST = True, same output paths.
# Only change: REGISTRY gains 9 T4 entries.
# T4 healthy uses session2 only (matched HOME with anomalies). Session1 is documented

import os, glob, h5py
import numpy as np
import pandas as pd
import scipy.stats as sp_stats

ROOT     = r"D:\Research\RCM"
LAB_DATA = os.path.join(ROOT, "Lab_Data")
OUT_DIR  = os.path.join(ROOT, "Processed_Data")
FIG_SUP  = os.path.join(ROOT, "Manuscript_Data", "Figures", "Supplementary")

RATE_HZ           = 125
MIN_CYCLE_SAMPLES = 200
TRIM_FIRST_LAST   = True

for d in [OUT_DIR, FIG_SUP]:
    os.makedirs(d, exist_ok=True)

JOINT_NAMES = [
    "J0_base", "J1_shoulder", "J2_elbow",
    "J3_wrist1", "J4_wrist2", "J5_wrist3",
]

# T4 entries appended at the bottom. session2 healthy only (matched HOME).

REGISTRY = {
    "T1_healthy":    ("T1_PickPlace/Healthy",  "UR5_T1_healthy_180cyc_*.h5",        "T1", "healthy", "none",    0.0),
    "T2_healthy":    ("T2_Assembly/Healthy",   "UR5_T2_healthy_180cyc_*.h5",        "T2", "healthy", "none",    0.0),
    "T3_healthy":    ("T3_Palletize/Healthy",  "UR5_T3_healthy_183cyc_*.h5",        "T3", "healthy", "none",    0.0),
    "T1_A2_0.5kg":   ("T1_PickPlace/A2", "UR5_T1_A2_0.5kg_gripper_40cyc_*.h5",  "T1", "A2", "0.5kg",   0.5),
    "T1_A2_1kg":     ("T1_PickPlace/A2", "UR5_T1_A2_1kg_gripper_40cyc_*.h5",    "T1", "A2", "1kg",     1.0),
    "T1_A2_2kg":     ("T1_PickPlace/A2", "UR5_T1_A2_2kg_gripper_40cyc_*.h5",    "T1", "A2", "2kg",     2.0),
    "T1_A3_10wraps": ("T1_PickPlace/A3", "UR5_T1_A3_1band_40cyc_*.h5",          "T1", "A3", "10wraps", 0.0),
    "T1_A3_17wraps": ("T1_PickPlace/A3", "UR5_T1_A3_3bands_40cyc_*.h5",         "T1", "A3", "17wraps", 0.0),
    "T1_A5_20mm":    ("T1_PickPlace/A5", "UR5_T1_A5_20mm_40cyc_*.h5",           "T1", "A5", "20mm",    0.0),
    "T1_A5_50mm":    ("T1_PickPlace/A5", "UR5_T1_A5_50mm_40cyc_*.h5",           "T1", "A5", "50mm",    0.0),
    "T1_A5_100mm":   ("T1_PickPlace/A5", "UR5_T1_A5_100mm_40cyc_*.h5",          "T1", "A5", "100mm",   0.0),
    "T2_A2_1.5kg":   ("T2_Assembly/A2", "UR5_T2_A2_1.5kg_gripper_40cyc_*.h5",   "T2", "A2", "0.5kg",   0.5),
    "T2_A2_2kg":     ("T2_Assembly/A2", "UR5_T2_A2_2kg_gripper_40cyc_*.h5",     "T2", "A2", "1kg",     1.0),
    "T2_A2_3kg":     ("T2_Assembly/A2", "UR5_T2_A2_3kg_gripper_40cyc_*.h5",     "T2", "A2", "2kg",     2.0),
    "T2_A3_7duct":   ("T2_Assembly/A3", "UR5_T2_A3_light_duct_40cyc_*_214735.h5",  "T2", "A3", "7wraps",  0.0),
    "T2_A3_14duct":  ("T2_Assembly/A3", "UR5_T2_A3_medium_duct_40cyc_*_225508.h5", "T2", "A3", "14wraps", 0.0),
    "T2_A5_20mm":    ("T2_Assembly/A5", "UR5_T2_A5_20mm_40cyc_*.h5",            "T2", "A5", "20mm",    0.0),
    "T2_A5_50mm":    ("T2_Assembly/A5", "UR5_T2_A5_50mm_40cyc_*.h5",            "T2", "A5", "50mm",    0.0),
    "T2_A5_100mm":   ("T2_Assembly/A5", "UR5_T2_A5_100mm_40cyc_*.h5",           "T2", "A5", "100mm",   0.0),
    "T3_A2_3.5kg":   ("T3_Palletize/A2", "UR5_T3_A2_3.5kg_gripper_33cyc_*.h5",  "T3", "A2", "0.5kg",   0.5),
    "T3_A2_4kg":     ("T3_Palletize/A2", "UR5_T3_A2_4kg_gripper_33cyc_*.h5",    "T3", "A2", "1kg",     1.0),
    "T3_A2_5kg":     ("T3_Palletize/A2", "UR5_T3_A2_4.5kg_gripper_33cyc_*.h5",  "T3", "A2", "2kg",     2.0),
    "T3_A3_14duct":  ("T3_Palletize/A3", "UR5_T3_A3_medium_duct_33cyc_*.h5",    "T3", "A3", "14wraps", 0.0),
    "T3_A3_7duct":   ("T3_Palletize/A3", "UR5_T3_A3_light_duct_33cyc_*.h5",     "T3", "A3", "7wraps",  0.0),
    "T3_A5_20mm":    ("T3_Palletize/A5", "UR5_T3_A5_20mm_33cyc_*.h5",           "T3", "A5", "20mm",    0.0),
    "T3_A5_50mm":    ("T3_Palletize/A5", "UR5_T3_A5_50mm_33cyc_*.h5",           "T3", "A5", "50mm",    0.0),
    "T3_A5_100mm":   ("T3_Palletize/A5", "UR5_T3_A5_100mm_33cyc_*.h5",          "T3", "A5", "100mm",   0.0),

    # T4 entries (new). Lowercase folder casing per NB1d.
    # Healthy: session2 ONLY (matched HOME with anomalies).
    "T4_healthy":    ("T4_BinReorient/healthy", "UR5_T4_healthy_session2_35cyc_*.h5", "T4", "healthy", "none",    0.0),
    "T4_A2_0.5kg":   ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",        "T4", "A2", "0.5kg",   0.5),
    "T4_A2_1kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",          "T4", "A2", "1kg",     1.0),
    "T4_A2_2kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",          "T4", "A2", "2kg",     2.0),
    "T4_A3_7duct":   ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",       "T4", "A3", "7wraps",  0.0),
    "T4_A3_14duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",      "T4", "A3", "14wraps", 0.0),
    "T4_A5_20mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",         "T4", "A5", "20mm",    0.0),
    "T4_A5_50mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",         "T4", "A5", "50mm",    0.0),
    "T4_A5_100mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",        "T4", "A5", "100mm",   0.0),
}

SEVERITY_ORDER = {
    "none": 0,
    "0.5kg": 1, "1kg": 2, "2kg": 3,
    "7wraps": 1, "10wraps": 2, "14wraps": 3, "17wraps": 4,
    "20mm": 1, "50mm": 2, "100mm": 3,
}

print(f"Registry: {len(REGISTRY)} canonical files (T4 added: 9)")

paths = {}
missing = []
for tag, (subdir, pattern, *_rest) in REGISTRY.items():
    full_glob = os.path.join(LAB_DATA, subdir, pattern)
    matches = sorted(glob.glob(full_glob))
    if len(matches) == 0:
        missing.append((tag, full_glob))
        continue
    if len(matches) > 1:
        # Take most recent file by mtime
        matches.sort(key=lambda p: os.path.getmtime(p))
        paths[tag] = matches[-1]
        print(f"  {tag}: {len(matches)} matches, using newest: {os.path.basename(matches[-1])}")
    else:
        paths[tag] = matches[0]

if missing:
    print(f"\n  MISSING ({len(missing)} entries):")
    for tag, g in missing:
        print(f"    {tag}: {g}")

print(f"\nResolved: {len(paths)}/{len(REGISTRY)} files")

def cycles_from_labels(cycle_arr):
    out, cur, s = [], -1, 0
    for i in range(len(cycle_arr)):
        if cycle_arr[i] != cur:
            if cur >= 0:
                out.append((s, i, int(cur)))
            cur = cycle_arr[i]
            s = i
    if cur >= 0:
        out.append((s, len(cycle_arr), int(cur)))
    return out

def cycles_from_tcp(tcp, home_r_mm=15.0, far_mm=30.0, min_n=500):
    home_pos = tcp[0, :3]
    dist_mm = np.sqrt(np.sum((tcp[:, :3] - home_pos)**2, axis=1)) * 1000.0
    out = []
    s, was_far, c = 0, False, 0
    for i in range(1, len(tcp)):
        if dist_mm[i] > far_mm:
            was_far = True
        if was_far and dist_mm[i] < home_r_mm and (i - s) > min_n:
            out.append((s, i, c))
            c += 1
            s = i
            was_far = False
    return out

def cycle_features(seg):
    """Compute 110-dim feature vector for a (N, 6) current segment."""
    f = {"n_samples": seg.shape[0], "duration_sec": seg.shape[0] / RATE_HZ}

    for j in range(6):
        p = JOINT_NAMES[j]
        s = seg[:, j]
        d = np.diff(s)

        f[f"{p}_mean"]         = np.mean(s)
        f[f"{p}_std"]          = np.std(s)
        f[f"{p}_min"]          = np.min(s)
        f[f"{p}_max"]          = np.max(s)
        f[f"{p}_range"]        = np.ptp(s)
        f[f"{p}_rms"]          = np.sqrt(np.mean(s**2))
        f[f"{p}_abs_mean"]     = np.mean(np.abs(s))
        f[f"{p}_skew"]         = float(sp_stats.skew(s))
        f[f"{p}_kurtosis"]     = float(sp_stats.kurtosis(s))
        f[f"{p}_p05"]          = np.percentile(s, 5)
        f[f"{p}_p25"]          = np.percentile(s, 25)
        f[f"{p}_p50"]          = np.percentile(s, 50)
        f[f"{p}_p75"]          = np.percentile(s, 75)
        f[f"{p}_p95"]          = np.percentile(s, 95)
        f[f"{p}_iqr"]          = f[f"{p}_p75"] - f[f"{p}_p25"]
        f[f"{p}_diff_std"]     = np.std(d)
        f[f"{p}_diff_abs_mean"]= np.mean(np.abs(d))

    f["total_rms"]     = np.sqrt(np.mean(seg**2))
    f["total_abs_max"] = np.max(np.abs(seg))
    return f

n_feats = len(cycle_features(np.random.randn(1000, 6)))
print(f"Features per cycle: {n_feats}")

rows, summaries = [], []

for tag in sorted(paths):
    subdir, pattern, task, anomaly, severity, extra_mass = REGISTRY[tag]
    fp = paths[tag]

    with h5py.File(fp, "r") as f:
        cur = f["actual_current"][:]
        if "cycle_number" in f:
            segs = cycles_from_labels(f["cycle_number"][:])
            method = "labels"
        else:
            segs = cycles_from_tcp(f["actual_TCP_pose"][:],
                                   home_r_mm=float(f.attrs.get("home_radius_mm", 15.0)))
            method = "tcp"

    if TRIM_FIRST_LAST and len(segs) > 2:
        segs = segs[1:-1]

    n_ok = 0
    for s, e, cn in segs:
        if (e - s) < MIN_CYCLE_SAMPLES:
            continue
        feat = cycle_features(cur[s:e])
        feat.update(dict(
            tag=tag, task=task, anomaly=anomaly, severity=severity,
            severity_order=SEVERITY_ORDER.get(severity, 0),
            extra_mass_kg=extra_mass,
            is_anomaly=int(anomaly != "healthy"),
            cycle_num=cn, file=os.path.basename(fp),
        ))
        rows.append(feat)
        n_ok += 1

    summaries.append(dict(tag=tag, task=task, anomaly=anomaly,
                          severity=severity, cycles_used=n_ok, method=method))
    print(f"  {tag:<20s}  {n_ok:>4d} cycles  ({method})")

print(f"\nTotal: {len(rows)} cycle-feature vectors from {len(paths)} files")

df = pd.DataFrame(rows)

for col in df.select_dtypes(include=["string", "object", "category"]).columns:
    df[col] = df[col].astype("object")

meta = ["tag", "task", "anomaly", "severity", "severity_order",
        "extra_mass_kg", "is_anomaly", "cycle_num", "file",
        "n_samples", "duration_sec"]
feats = sorted(c for c in df.columns if c not in meta)
df = df[meta + feats]

csv_path = os.path.join(OUT_DIR, "features.csv")
h5_path  = os.path.join(OUT_DIR, "features.h5")

df.to_csv(csv_path, index=False, float_format="%.6f")
df.to_hdf(h5_path, key="features", mode="w", format="table")

print(f"\nSaved: {csv_path} ({os.path.getsize(csv_path)/1e6:.2f} MB)")
print(f"Saved: {h5_path}")

summary_df = pd.DataFrame(summaries)
print("\nCycle-count summary:")
print(summary_df.to_string(index=False))

print("\nPer-task healthy cycles:")
print(df[df["is_anomaly"] == 0].groupby("task")["cycle_num"].count())

print("\nPer-task anomaly cycles:")
print(df[df["is_anomaly"] == 1].groupby(["task", "anomaly"])["cycle_num"].count())

t4_h = df[(df["task"] == "T4") & (df["is_anomaly"] == 0)]
t4_a = df[(df["task"] == "T4") & (df["is_anomaly"] == 1)]
print(f"\nT4 sanity:")
print(f"  T4 healthy cycles: {len(t4_h)}")
print(f"  T4 anomaly cycles: {len(t4_a)} across {t4_a['anomaly'].nunique()} anomaly types and {t4_a['severity'].nunique()} severities")

# Confirm all 4 tasks present
print(f"\nTasks present in features.csv: {sorted(df['task'].unique())}")

print("\nNB7d complete. features.csv and features.h5 now contain all four tasks.")
print("Next step: run the existing NB8 (PSR), then NB10* downstream notebooks. They auto-pickup T4.")